<a href="https://colab.research.google.com/github/rlagosb/GastricCancerIncidence/blob/main/1_Participants_and_trajectories.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook presents an analysis of gastric cancer incidence, the demographic characteristics of the study and target population, patient trajectories, and in-hospital survival. The objective is to provide detailed information for researchers interested in the epidemiology of gastric cancer and its outcomes.

The content includes:

*   **Data loading and preparation:** Data on populations, hospital discharges, and deaths related to digestive cancer are loaded and processed.
*   **Characterization of participants (Table 1):** A descriptive table of the study and target population is presented, stratified by sex, age ranges, rurality, and type of health insurance (Fonasa/Isapre).
*   **Gastric Cancer Trajectories:** Sequences of events (discharge, death) for patients are analyzed, including a Sankey diagram to visualize transition flows.
*   **Trajectories by province:** Deaths without hospitalization and 5-year survival by province and five-year period are examined to provide a regional perspective of the results.

# Setup

In [5]:
import pandas as pd
import plotly.graph_objects as go

path_cubo = 'https://raw.githubusercontent.com/rlagosb/GastricCancerIncidence/main/Data/'

# load main table
cubo = pd.read_parquet(path_cubo + 'CUBO_CANCER_DIGESTIVO.parquet')
cubo = cubo[cubo.Provincia!=122] #excluir provincia Antártica
print(cubo.info(), end='\n\n')

# load provinces
provincias = (cubo[['Region','Provincia','Nombre Region','Nombre Provincia']].
              drop_duplicates())

# Cargar egresos-defunciones desagregados
egresosDefs = pd.read_parquet(path_cubo + 'EGRESOS_DEFUNCIONES_DESAGREGADOS.parquet')
egresosDefs = egresosDefs[egresosDefs.Provincia!=122] #excluir provincia Antártica
print(egresosDefs.info())

<class 'pandas.core.frame.DataFrame'>
Index: 24156 entries, 0 to 24947
Data columns (total 26 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Provincia             24156 non-null  int64  
 1   Nombre Provincia      24156 non-null  object 
 2   Nombre Region         24156 non-null  object 
 3   Region                24156 non-null  int64  
 4   Macrorregion          24156 non-null  object 
 5   Año                   24156 non-null  int64  
 6   Sexo                  24156 non-null  object 
 7   RangoEdad10           24156 non-null  object 
 8   RangoEdad4080         24156 non-null  object 
 9   MedianaRangoEdad10    24156 non-null  float64
 10  MedianaRangoEdad4080  24156 non-null  float64
 11  Poblacion             24156 non-null  int64  
 12  PoblacionRural        24156 non-null  int64  
 13  A                     24156 non-null  int64  
 14  B                     24156 non-null  int64  
 15  C                     24

# Participants
Study and objetive populations in person-years

In [6]:
def get_participantes(periodo_objetivo=range(2003,2025)):

  # Agrupar edad por rango
  metricas = ['Poblacion','PoblacionRural','Fonasa','A','B','C','D','Isapre']
  df = cubo.groupby(['Año','Sexo','RangoEdad4080','RPC','Provincia'],observed=True)[metricas].sum().reset_index()

  # Generate study (1) and objective (2) populations
  df1 = df[df.RPC>0].copy()
  df2 = df[df.Año.isin(periodo_objetivo)].copy()
  df1['Grupo'] = 'Estudio'
  df2['Grupo'] = 'Objetivo'
  df = pd.concat([df1,df2])

  # Contar Años
  agg_dict = {metric: 'sum' for metric in metricas}
  agg_dict['Año'] = 'count'
  df = (df.groupby(['Sexo', 'RangoEdad4080','RPC','Provincia','Grupo'], observed=True).agg(agg_dict).reset_index().
        rename(columns={'Año':'N_Años'}))

  return df

poblaciones = get_participantes()
poblaciones

,Sexo,RangoEdad4080,RPC,Provincia,Grupo,Poblacion,PoblacionRural,Fonasa,A,B,C,D,Isapre,N_Años
0,Hombre,"[0, 40)",0,11,Objetivo,2209190,34920,1453505,405604,340120,302172,405609,542627,22
1,Hombre,"[0, 40)",0,14,Objetivo,175192,81561,122398,50237,23944,18164,30277,13795,22
2,Hombre,"[0, 40)",0,21,Objetivo,1287422,28215,773474,144760,190287,154054,284451,355918,9
3,Hombre,"[0, 40)",0,22,Objetivo,567792,31083,377674,55864,88600,73766,159624,155859,9
4,Hombre,"[0, 40)",0,23,Objetivo,92156,4668,43295,15223,7825,6584,13663,17205,9
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1003,Mujer,"[80, 120)",6,72,Objetivo,5087,1336,4332,27,3955,149,201,51,3
1004,Mujer,"[80, 120)",6,73,Estudio,14774,3929,7950,53,11740,517,582,320,3
1005,Mujer,"[80, 120)",6,73,Objetivo,14774,3929,7950,53,11740,517,582,320,3
1006,Mujer,"[80, 120)",6,74,Estudio,16191,5496,13381,155,13499,564,583,277,3


## 📊 Table 1

Based on recomendations of Hayes-Larson, Eleanor, Katrina L. Kezios, Stephen J. Mooney, y Gina Lovasi. «Who Is in This Study, Anyway? Guidelines for a Useful Table 1». Journal of Clinical Epidemiology 114 (octubre de 2019): 125-32. https://doi.org/10.1016/j.jclinepi.2019.06.011.


In [8]:
def subtabla(variable, RangoEdad, df) -> pd.DataFrame:

  # sumar total población

  match variable:
    case 'Total':
      df = df.groupby(['Grupo'],observed=True)['Poblacion'].sum().reset_index()
      df['Total'] = df['Grupo'] + ' N=' + df['Poblacion'].astype(str)
      return df.Total.to_list()

    case 'Sexo' | 'RangoEdad4080':
      df = df.pivot_table(index=variable, values='Poblacion',columns='Grupo',aggfunc='sum',margins=True)
      # transform to percentages
      df['Estudio'] = df['Estudio']/df.loc['All','Estudio']
      df['Objetivo'] = df['Objetivo']/df.loc['All','Objetivo']
      return df.drop('All').drop(columns='All').reset_index(names=variable)

    case 'Ruralidad':
      df = df.groupby(['Grupo'], observed=True)[['PoblacionRural','Poblacion']].sum().reset_index()
      df['Ruralidad'] = df['PoblacionRural']/df['Poblacion']
      return df[['Grupo','Ruralidad']].set_index('Grupo').T

    case 'Beneficiarios':
      tramos = ['A','B','C','D','Isapre']
      df = df.groupby(['Grupo'], observed=True)[['Poblacion'] + tramos].sum().reset_index()
      for col in tramos:
        df[col] = df[col]/df['Poblacion']
      return df.set_index('Grupo').T.drop('Poblacion').reset_index(names='Beneficiarios')


def get_tabla1(RangoEdad, df):

  # Agrupar tablas
  tabla = pd.DataFrame()
  for variable in ['Sexo',RangoEdad,'Ruralidad','Beneficiarios']:
    tabla_var = subtabla(variable, RangoEdad, df)
    tabla_var['Variable'] = variable
    tabla_var.rename(columns={variable:'Valor'},inplace=True)
    tabla = pd.concat([tabla, tabla_var])

  # Formatear índice y columnas
  tabla.set_index(['Variable','Valor'], inplace=True)
  tabla.columns = subtabla('Total', 'RangoEdad4080', poblaciones)

  return (tabla*100).round(1)


tabla1 = get_tabla1('RangoEdad4080', poblaciones)
tabla1


Estudio N=29348519  Objetivo N=394057268
Variable      Valor                                              
Sexo          Hombre                   49.6                  49.3
              Mujer                    50.4                  50.7
RangoEdad4080 [0, 40)                  61.9                  59.8
              [40, 50)                 13.9                  13.7
              [50, 60)                 11.1                  11.5
              [60, 70)                  7.1                   8.0
              [70, 80)                  4.1                   4.7
              [80, 120)                 2.0                   2.4
Ruralidad     NaN                      15.5                  12.3
Beneficiarios A                        18.7                  19.6
              B                        20.4                  26.4
              C                        10.1                  12.5
              D                        12.1                  16.4
              Isapre                   17.3                  17.9

# Trajectories

In [9]:
#@title Cargar Eventos

# Eventos sin id Persona o Provincia
egresosDefs['SinIdPersona'] = (egresosDefs.idPersona.isna())
egresosDefs['SinProvincia'] = (egresosDefs.Provincia==999)

print(egresosDefs.reset_index().pivot_table(index='Categoria',
                                      columns=['Trayectoria','SinProvincia','SinIdPersona'],
                                      values='index',
                                      aggfunc='count',
                                      margins=True))

# Filtrar eventos sin id Persona o Provincia
egresosDefs = egresosDefs[lambda x: ~x.SinProvincia & ~x.SinIdPersona]

Trayectoria  Defuncion  Egreso           All
SinProvincia     False   False              
SinIdPersona     False   False  True        
Categoria                                   
C16              69969  107529  2352  179850
All              69969  107529  2352  179850


In [10]:
#@title Cargar Trayectorias de Personas

def cargar_trayectorias_personas(df):
  # Concatenar eventos CG de una persona
  def serie_trayectorias(serie):
    serie = serie.astype(str) + '>'
    x = serie.cumsum()
    return x.iloc[-1] # Return only the last cumulative trajectory

  trayectorias = (df.groupby(['idPersona'], dropna=True)['Trayectoria'].
                  apply(serie_trayectorias).str[:-1].  # agrupar eventos de cada persona
                  value_counts(dropna=False))
  return trayectorias.reset_index()

trayectorias = cargar_trayectorias_personas(egresosDefs.copy())

print('Top 5 trayectorias\n:', trayectorias.head(5), end='\n\n')

print(trayectorias['count'].sum(), 'personas con trayectorias de CG,\n',
      trayectorias[lambda x: x.Trayectoria.str.contains('Defuncion')]['count'].sum()*100/trayectorias['count'].sum(), '% con defunción\n',
      trayectorias[lambda x: x.Trayectoria.str.contains('Egreso')]['count'].sum()*100/trayectorias['count'].sum(), '% con egreso\n',
      trayectorias[lambda x: x.Trayectoria.str.contains('Defuncion') & ~x.Trayectoria.str.contains('Egreso')]['count'].sum()*100/trayectorias['count'].sum(), '% con defunción sin egreso')


Top 5 trayectorias
:                Trayectoria  count
0                Defuncion  33191
1         Egreso>Defuncion  23134
2                   Egreso  19572
3  Egreso>Egreso>Defuncion   7698
4            Egreso>Egreso   4188

96187 personas con trayectorias de CG,
 72.74059904145051 % con defunción
 65.49117864160438 % con egreso
 34.50882135839562 % con defunción sin egreso


In [11]:
#@title 🔀 Figura 1

def diagrama_sankey(trayectorias):

  # Parametrizar transiciones
  for col in ['Disease-free_Hospitalized','Disease-free_Deceased','Hospitalized_Alive','Hospitalized_Deceased']:
    trayectorias[col] = False
  egresoInicio = (trayectorias.Trayectoria.str[:6]=='Egreso')
  conDefuncion = (trayectorias.Trayectoria.str.contains('Defuncion'))
  # Sano>Egreso
  trayectorias.loc[egresoInicio,'Disease-free_Hospitalized'] = True
  # Sano>Defuncion
  trayectorias.loc[trayectorias.Trayectoria.str[:9]=='Defuncion','Disease-free_Deceased'] = True
  # Egreso>Vivo
  trayectorias.loc[egresoInicio & ~conDefuncion,'Hospitalized_Alive'] = True
  # Egreso>Defuncion
  trayectorias.loc[egresoInicio & conDefuncion,'Hospitalized_Deceased'] = True

  arcos = (trayectorias.melt(id_vars=['Trayectoria','count'],
                           value_vars=['Disease-free_Hospitalized','Disease-free_Deceased','Hospitalized_Alive','Hospitalized_Deceased'],
                           var_name='Arco',value_name='Valor')[lambda x: x.Valor].  # dejar sólo transiciones activas
           groupby(['Arco','Valor'])['count'].sum().reset_index())                  # agrupar transiciones
  # separar origen/destino
  arcos[['Source','Target']] = (arcos.Arco.str.split('_',expand=True))

  # Create Sankey diagram https://plotly.com/python/sankey-diagram/
  nodes = pd.concat([arcos['Source'], arcos['Target']]).unique()
  nodes = pd.DataFrame({'name': nodes})
  nodes['id'] = nodes.index
  arcos = arcos.merge(nodes.rename(columns={'name':'Source'}), on='Source', how='left')
  arcos = arcos.merge(nodes.rename(columns={'name':'Target'}), on='Target', how='left', suffixes=('_source', '_target'))

  go.Figure(data=[go.Sankey(
      node=dict(pad=15, thickness=20,
          line=dict(color="black", width=0.5),
          label=nodes['name'],),
      link=dict(
          source=arcos['id_source'],
          target=arcos['id_target'],
          value=arcos['count']))]).show()


diagrama_sankey(trayectorias.copy())

# Trayectorias Provincias

In [12]:
def trayectorias_persona(df):
  df = df[df.idPersona.notna()]

  # Obtener año de egreso y hospitalización por persona
  fa = df[df.Trayectoria=='Defuncion'][['idPersona','Año']].rename(columns={'Año':'AñoDefuncion'})
  eg = df[df.Trayectoria=='Egreso'][['idPersona','Año']].rename(columns={'Año':'AñoEgreso'})
  df = (df.groupby(['idPersona']).agg({'Provincia':'first'}).reset_index().
        merge(eg, on='idPersona', how='outer').
        merge(fa, on='idPersona', how='outer').
        groupby(['idPersona','Provincia']).agg({'AñoEgreso':'min','AñoDefuncion':'max'}).reset_index())

  # Métricas sobrevida
  df['Delta'] = df.AñoDefuncion - df.AñoEgreso
  df.loc[df.AñoDefuncion.isna(), 'sobrevida5'] = True
  df.loc[df.Delta.notna(), 'sobrevida5'] = df[df.Delta.notna()].Delta.apply(lambda x: x>=5)

  # get minimum between AñoEgreso and AñoDefuncion
  df['Quinquenio'] = df[['AñoEgreso','AñoDefuncion']].min(axis=1).apply(lambda x: (x//5)*5)

  df['Defuncion'] = df.AñoDefuncion.notna()
  df['Egreso'] = df.AñoEgreso.notna()
  df['DefuncionSinEgreso'] = df.Defuncion & ~df.Egreso

  df = df.groupby(['Provincia','Quinquenio']).agg({'Delta':'mean','sobrevida5':'sum','Defuncion':'sum','Egreso':'sum','DefuncionSinEgreso':'sum','idPersona':'count'}).reset_index()

  return df

trayectoriasProv = trayectorias_persona(egresosDefs.copy())

## 📊 Defunciones sin hospitalización (Material Sup 3)

In [14]:
# Defunción sin hospitalización por provincia y quinquenio

print('Porcentaje de defunciones sin hospitalización:\n',
      trayectoriasProv.DefuncionSinEgreso.sum()*100/trayectoriasProv.idPersona.sum())

trayectoriasProv['PorcDefSinEgreso'] = trayectoriasProv.DefuncionSinEgreso*100/trayectoriasProv.idPersona

trayectoriasProv[lambda x: x.Quinquenio>2000].\
pivot(columns='Quinquenio',index='Provincia',values='PorcDefSinEgreso').\
merge(provincias, on='Provincia').\
sort_values(by=['Region','Provincia']).\
set_index(['Region','Nombre Region','Nombre Provincia']).drop(columns='Provincia').round(1)

Porcentaje de defunciones sin hospitalización:
 34.54042185662702


2005.0  \
Region Nombre Region                             Nombre Provincia               
1      Tarapacá                                  Iquique                 67.0   
                                                 Tamarugal               77.8   
2      Antofagasta                               Antofagasta             32.1   
                                                 El Loa                  29.0   
                                                 Tocopilla               16.0   
3      Atacama                                   Copiapó                 34.6   
                                                 Chañaral                23.3   
                                                 Huasco                  28.2   
4      Coquimbo                                  Elqui                   40.3   
                                                 Choapa                  51.3   
                                                 Limarí                  39.2   
5      Valparaíso                                Valparaíso              25.3   
                                                 Isla de Pascua          25.0   
                                                 Los Andes               33.3   
                                                 Petorca                 37.4   
                                                 Quillota                34.9   
                                                 San Antonio             38.5   
                                                 San Felipe              14.7   
                                                 Marga Marga             37.4   
6      Libertador General Bernardo O'Higgins     Cachapoal               34.7   
                                                 Cardenal Caro           45.8   
                                                 Colchagua               38.1   
7      Maule                                     Talca                   43.9   
                                                 Cauquenes               54.0   
                                                 Curicó                  40.0   
                                                 Linares                 41.5   
8      Biobío                                    Concepción              28.8   
                                                 Arauco                  33.4   
                                                 Bíobío                  33.2   
9      La Araucanía                              Cautín                  38.0   
                                                 Malleco                 28.7   
10     Los Lagos                                 Llanquihue              29.0   
                                                 Chiloé                  45.1   
                                                 Osorno                  37.0   
                                                 Palena                  30.6   
11     Aysén del General Carlos Ibáñez del Campo Coyhaique               38.6   
                                                 Aysén                   43.9   
                                                 Capitán Prat            60.0   
                                                 General Carrera         38.5   
12     Magallanes y de la Antártica Chilena      Magallanes              30.8   
                                                 Tierra del Fuego        20.0   
                                                 Última Esperanza        50.0   
13     Metropolitana de Santiago                 Cordillera              24.3   
                                                 Chacabuco               39.6   
                                                 Maipo                   36.9   
                                                 Melipilla               51.5   
                                                 Talagante               52.9   
                                                 Santiago Norte          32.8   
                     

## Sobrevida Intrahospitalaria (Material Sup 4)


In [16]:
# add colum percentage sobrevida5

# Agregar total nacional
sobre = pd.concat([trayectoriasProv.copy(),
                  trayectoriasProv.groupby(['Quinquenio'])[['sobrevida5','idPersona']].sum().reset_index()])
# Excluir último quinquenio
sobre = sobre[sobre.Quinquenio<2020]

# Calcular sobrevida 5 años
sobre['PorcSobrevida5'] = (sobre.sobrevida5*100.0/sobre.idPersona).round(1)

print(f'Sobrevida a 5 años nacional:\n {sobre[sobre.Provincia.isna()].sobrevida5.sum()/sobre[sobre.Provincia.isna()].idPersona.sum()*100} %')

# Calcular sobrevida por provincia
sobre = sobre.pivot(index='Provincia', columns='Quinquenio', values='PorcSobrevida5').\
merge(provincias, on='Provincia', how='left').\
sort_values(by=['Region','Provincia']).\
drop(columns='Provincia').set_index(['Region','Nombre Region','Nombre Provincia'])

for col in sobre.columns:
  sobre[col] = sobre[col].astype(float).round(1)

pd.set_option('display.max_rows', None)
display(sobre)

Sobrevida a 5 años nacional:
 26.565981348311606 %


2000.0  \
Region Nombre Region                             Nombre Provincia               
1.0    Tarapacá                                  Iquique                 14.1   
                                                 Tamarugal               33.3   
2.0    Antofagasta                               Antofagasta             23.7   
                                                 El Loa                  36.8   
                                                 Tocopilla               11.1   
3.0    Atacama                                   Copiapó                 40.0   
                                                 Chañaral                52.9   
                                                 Huasco                  12.8   
4.0    Coquimbo                                  Elqui                   26.3   
                                                 Choapa                  22.2   
                                                 Limarí                  20.4   
5.0    Valparaíso                                Valparaíso              25.6   
                                                 Isla de Pascua           NaN   
                                                 Los Andes               32.0   
                                                 Petorca                 25.8   
                                                 Quillota                27.1   
                                                 San Antonio             15.3   
                                                 San Felipe              24.7   
                                                 Marga Marga             35.2   
6.0    Libertador General Bernardo O'Higgins     Cachapoal               20.5   
                                                 Cardenal Caro           29.4   
                                                 Colchagua               17.4   
7.0    Maule                                     Talca                   16.4   
                                                 Cauquenes               10.0   
                                                 Curicó                  17.1   
                                                 Linares                 16.8   
8.0    Biobío                                    Concepción              41.2   
                                                 Arauco                  38.6   
                                                 Bíobío                  23.2   
9.0    La Araucanía                              Cautín                  17.5   
                                                 Malleco                 20.9   
10.0   Los Lagos                                 Llanquihue              22.7   
                                                 Chiloé                  14.8   
                                                 Osorno                  26.7   
                                                 Palena                  57.1   
11.0   Aysén del General Carlos Ibáñez del Campo Coyhaique               16.7   
                                                 Aysén                   41.2   
                                                 Capitán Prat            50.0   
                                                 General Carrera          0.0   
12.0   Magallanes y de la Antártica Chilena      Magallanes              25.4   
                                                 Tierra del Fuego        12.5   
                                                 Última Esperanza        12.5   
13.0   Metropolitana de Santiago                 Cordillera              35.6   
                                                 Chacabuco               26.1   
                                                 Maipo                   21.9   
                                                 Melipilla               23.9   
                                                 Talagante               18.8   
                                                 Santiago Norte          30.3   
                     